# AWI-ESM-1-1-LR

I don't think this will work, unfortunately. There is data missing at every spatial location, meaning that when the NaNs are propagated, the whole dataset becomes NaNs.

In [31]:
import xarray as xr
import os
import numpy as np
from eofs.standard import Eof
import logging

from functions.SIT_functions.SIT_eddy_feedback_functions import eof_calc, vert_integrate, read_daily_averages, propagate_missing_data_to_all_vars

In [32]:
exp_type = 'cmip'
output_eof_file = '/home/users/cturrell/documents/eddy_feedback/b-parameter/cmip6_b/hist_NaN_issues/data'
force_eof_recalculate = True
pfull_slice = slice(850., 100.)
subtract_annual_cycle=True
eof_vars = ['ucomp', 'div1_QG', 'div1_QG_123', 'div1_QG_gt3']
n_eofs = 3
season_month_dict = {'DJF':[12,1,2,]}
propogate_all_nans = True

In [33]:
awi_path = '/gws/ssde/j25a/arctic_connect/cturrell/CMIP6/historical/AWI-ESM-1-1-LR/1850_2015/6hrPlevPt/'
anom_ds_awi = xr.open_dataset(os.path.join(awi_path, '1979_2015', 'anoms.nc'))
dataset_awi = read_daily_averages(os.path.join(awi_path, 'yearly_data'), start_month=1979, end_month=2015, exp_type='cmip6')

# eof_awi = eof_calc('cmip', output_eof_file, force_eof_recalculate, dataset_awi, pfull_slice,
#                               subtract_annual_cycle, eof_vars, n_eofs, season_month_dict, anom_ds_awi,
#                               propogate_all_nans, level_subset=[250., 500., 850.],
#                               pressure_weighted=True)

2026-04-15 11:21:47,583 [INFO]: SIT_eddy_feedback_functions.py(read_daily_averages:1135) >> reading daily average files
2026-04-15 11:21:47,583 [INFO]: SIT_eddy_feedback_functions.py(read_daily_averages:1135) >> reading daily average files
2026-04-15 11:21:48,994 [INFO]: SIT_eddy_feedback_functions.py(read_daily_averages:1176) >> finished reading daily average files
2026-04-15 11:21:48,994 [INFO]: SIT_eddy_feedback_functions.py(read_daily_averages:1176) >> finished reading daily average files
2026-04-15 11:21:49,017 [INFO]: SIT_eddy_feedback_functions.py(check_for_duplicate_times:1201) >> duplicate times are present!!!
2026-04-15 11:21:49,017 [INFO]: SIT_eddy_feedback_functions.py(check_for_duplicate_times:1201) >> duplicate times are present!!!
2026-04-15 11:21:49,321 [INFO]: SIT_eddy_feedback_functions.py(check_for_duplicate_times:1205) >> duplicate times removed. Was 13185 now 13150
2026-04-15 11:21:49,321 [INFO]: SIT_eddy_feedback_functions.py(check_for_duplicate_times:1205) >> dup

In [34]:
def propagate_missing_data_to_all_vars(anom_ds, eof_vars):
    '''The EOF projectfield method requires that each of the variables
    we project must have missing data in the same place as the field 
    we are projecting onto. This is quite difficult to ensure
    particularly when I have added nans to div1 and div2 when the dtheta/dp
    is vanishingly small. To get around this, this function looks where the 
    nan values are in each of the anomaly fields and makes sure every other 
    variable also has missing data there.'''

    anom_coord_list = [key for key in anom_ds.coords.keys()]
    # anom_var_list = [key for key in anom_ds.variables.keys() if key not in anom_coord_list and '_anom' in key]
    anom_var_list = [f'{key}_anom' for key in eof_vars]

    # nan_location_dict = {}
    n_nans_list = []

    for anom_var in anom_var_list:
        for anom_var_nan_var in anom_var_list:
            # Current — propagates transient NaNs
            anom_ds[anom_var] = anom_ds[anom_var].where(np.isfinite(anom_ds[anom_var_nan_var]))

            # DOESN'T WORK!!! Fixed — only propagates spatially-fixed NaN locations (NaN in ALL time steps)
            # fixed_mask = np.isfinite(anom_ds[anom_var_nan_var]).all('time')
            # anom_ds[anom_var] = anom_ds[anom_var].where(fixed_mask)

        n_nans_list.append(np.where(np.isnan(anom_ds[anom_var]))[0].shape[0])

    all_anom_vars_same_number_nan = np.all(np.asarray(n_nans_list)==n_nans_list[0])

    if all_anom_vars_same_number_nan:
        logging.info(f'successfully propagated missing data from all vars to all other vars to enable eof projection, so each field now has {n_nans_list[0]} nans')
    else:
        raise ValueError('anom vars contain differing number of nans, which means the projection will not work')

    return anom_ds

def obtain_orig_var_hem(exp_type, output_eof_file, force_eof_recalculate, dataset, pfull_slice,
             eof_vars, n_eofs, season_month_dict, anom_ds,
             propagate_all_nans, level_subset=None):  

    pfull_selector = level_subset if level_subset is not None else pfull_slice

    # Resolve level_subset to nearest actual levels in the data to avoid
    # exact-match failures on models with slightly different pressure coordinates
    if isinstance(pfull_selector, list):
        pfull_selector = [float(anom_ds['ucomp_anom'].sel(pfull=lev, method='nearest').pfull.values)
                          for lev in pfull_selector]

    if np.all(dataset.lat.diff('lat')<0.):
        hemisphere_slice_dict = {'n':slice(80.,10.),
                                's':slice(-10., -80.),                          
                                }
    else:
        hemisphere_slice_dict = {'n':slice(10.,80.),
                                's':slice(-80., -10.),                             
                                }        

    if propagate_all_nans:
        output_eof_file_use = output_eof_file.split('.nc')[0]+'_prop_nans.nc'
    else:
        output_eof_file_use = output_eof_file

    if os.path.isfile(output_eof_file_use) and not force_eof_recalculate:
        logging.info('attempting to read in EOF data')
        eof_ds = xr.open_mfdataset(output_eof_file_use, decode_times=True)
        logging.info('SUCCESS')
    else:
        logging.info('failed to read in previously calculated EOF data - CALCULATING')

        logging.info('calculating anomalies')
        if exp_type=='held_suarez':
            u_anoms = dataset['ucomp'].mean('lon') - dataset['ucomp'].mean(('time','lon'))
        else:
            # u_anoms_time = dataset.temporal.departures(data_var = 'ucomp', freq='day', weighted=False)['ucomp'].mean('lon')
            u_anoms_time = anom_ds['ucomp_anom']

        season_list = [season_val for season_val in season_month_dict.keys()]

        all_time_season_list = season_list+['all_time']

        eof_ds = xr.Dataset()
        eof_ds.coords['time'] = ('time', u_anoms_time.time.values)
        eof_ds.coords['pfull'] = ('pfull', u_anoms_time.sel(pfull=pfull_selector).pfull.values)  
        eof_ds.coords['lat'] = ('lat', u_anoms_time.sel(lat=hemisphere_slice_dict['n']).lat.values)                

        for season_val in season_list:
            u_anoms_season = u_anoms_time.where(anom_ds['time'].dt.month.isin(season_month_dict[season_val]), drop=True)
            eof_ds.coords[f'time_{season_val}'] = (f'time_{season_val}', u_anoms_season.time.values)

        eof_ds.coords['eof_num'] = ('eof_num', np.arange(n_eofs))

        ucomp_solver_dict = {}
        ucomp_500_solver_dict = {}        
        ucomp_va_solver_dict = {}        
        for season_val in all_time_season_list:
            ucomp_solver_dict[season_val] = {'n':{}, 's':{}}
            ucomp_va_solver_dict[season_val] = {'n':{}, 's':{}}
            ucomp_500_solver_dict[season_val] = {'n':{}, 's':{}}

        logging.info('about to propagate nans to all fields')
        if propagate_all_nans:
            anom_ds = propagate_missing_data_to_all_vars(anom_ds, eof_vars)

        for eof_var in eof_vars:

            logging.info('loading anomalies from anom_ds')

            var_anoms = anom_ds[f'{eof_var}_anom'].load()
            orig_var = anom_ds[f'{eof_var}_orig'].load()           

            for hemisphere in ['n','s']:

                for time_frame in all_time_season_list:
                    var_anoms_hem = var_anoms.sel(lat=hemisphere_slice_dict[hemisphere]).sel(pfull=pfull_selector)
                    
                    if time_frame!='all_time':
                        var_anoms_hem = var_anoms_hem.where(eof_ds['time'].dt.month.isin(season_month_dict[time_frame]), drop=True) 
                        time_dim_name = f'time_{time_frame}'
                    else:
                        time_dim_name = 'time'

                    orig_var_hem = orig_var.sel(lat=hemisphere_slice_dict[hemisphere]).sel(pfull=pfull_selector)
                    
                    if time_frame!='all_time':
                        orig_var_hem = orig_var_hem.where(eof_ds['time'].dt.month.isin(season_month_dict[time_frame]), drop=True)
                        
                    orig_var_hem = orig_var_hem.mean('time')   
                    
                    va_var_anoms_hem = vert_integrate(var_anoms_hem, pressure_weighted=True)

    return var_anoms_hem, va_var_anoms_hem, anom_ds

orig, va, anom_ds = obtain_orig_var_hem(exp_type, output_eof_file, force_eof_recalculate, dataset_awi, pfull_slice,
             eof_vars, n_eofs, season_month_dict, anom_ds_awi, propagate_all_nans=True, level_subset=[250.,500.,850.])
anom_ds

2026-04-15 11:21:49,406 [INFO]: 2342300006.py(obtain_orig_var_hem:68) >> failed to read in previously calculated EOF data - CALCULATING
2026-04-15 11:21:49,406 [INFO]: 2342300006.py(obtain_orig_var_hem:68) >> failed to read in previously calculated EOF data - CALCULATING
2026-04-15 11:21:49,407 [INFO]: 2342300006.py(obtain_orig_var_hem:70) >> calculating anomalies
2026-04-15 11:21:49,407 [INFO]: 2342300006.py(obtain_orig_var_hem:70) >> calculating anomalies
2026-04-15 11:21:49,562 [INFO]: 2342300006.py(obtain_orig_var_hem:100) >> about to propagate nans to all fields
2026-04-15 11:21:49,562 [INFO]: 2342300006.py(obtain_orig_var_hem:100) >> about to propagate nans to all fields
2026-04-15 11:21:52,080 [INFO]: 2342300006.py(propagate_missing_data_to_all_vars:31) >> successfully propagated missing data from all vars to all other vars to enable eof projection, so each field now has 210240 nans
2026-04-15 11:21:52,080 [INFO]: 2342300006.py(propagate_missing_data_to_all_vars:31) >> successfu

<xarray.Dataset> Size: 454MB
Dimensions:           (pfull: 6, lat: 96, time: 13141)
Coordinates:
  * pfull             (pfull) float64 48B 925.0 850.0 700.0 600.0 500.0 250.0
  * lat               (lat) float64 768B -88.57 -86.72 -84.86 ... 86.72 88.57
  * time              (time) datetime64[ns] 105kB 1979-01-01 ... 2015-01-01
Data variables:
    ucomp_anom        (time, pfull, lat) float64 61MB -0.9406 -1.17 ... 2.407
    ucomp_orig        (time, pfull, lat) float32 30MB -1.896 -2.723 ... 2.699
    div1_QG_anom      (time, pfull, lat) float64 61MB 1.131 0.7995 ... 6.906
    div1_QG_orig      (time, pfull, lat) float64 61MB 2.568 1.79 ... 3.249 7.388
    div1_QG_123_anom  (time, pfull, lat) float64 61MB 1.063 0.9048 ... 6.856
    div1_QG_123_orig  (time, pfull, lat) float64 61MB 2.435 2.032 ... 7.349
    div1_QG_gt3_anom  (time, pfull, lat) float64 61MB 0.06795 ... 0.04978
    div1_QG_gt3_orig  (time, pfull, lat) float64 61MB 0.133 -0.2417 ... 0.03924

In [35]:
print(anom_ds.ucomp_anom.size)
print(anom_ds.ucomp_anom.isnull().sum())

print((anom_ds.ucomp_anom.isnull().sum()/anom_ds.ucomp_anom.size)*100)

7569216
<xarray.DataArray 'ucomp_anom' ()> Size: 8B
array(210240)
<xarray.DataArray 'ucomp_anom' ()> Size: 8B
array(2.7775664)


In [36]:
subset = anom_ds.ucomp_anom.sel(lat=slice(10., 80.)).sel(pfull=[250., 500., 850.], method='nearest')

# How many spatial points are contaminated (have ANY NaN across time)?
n_contaminated = subset.isnull().any('time').sum().values
n_total_spatial = subset.sizes['lat'] * subset.sizes['pfull']
print(f"Contaminated spatial points: {n_contaminated} / {n_total_spatial}")

# The killer question:
print(f"ALL spatial points have ≥1 NaN: {subset.isnull().any('time').all().values}")


Contaminated spatial points: 114 / 114
ALL spatial points have ≥1 NaN: True


In [37]:
def eof_calc_alt(data,lats):

    coslat = np.cos(np.deg2rad(lats)).clip(0., 1.)
    wgts = np.sqrt(coslat)[np.newaxis, np.newaxis, :]

    solver = Eof(data, weights=wgts, center=True)

    eofs = solver.eofsAsCovariance(neofs=3)
    pc1 = solver.pcs(npcs=3, pcscaling=1)

    variance_fractions = solver.varianceFraction(neigs=3)

    return eofs, pc1, variance_fractions, solver

eofs_va, pc1_va, variance_fractions_va, solver_va = eof_calc_alt(anom_ds.ucomp_anom.values, anom_ds.ucomp_anom.lat.values)

ValueError: all input data is missing

In [ ]:
anom_ds.ucomp_anom

<xarray.DataArray 'ucomp_anom' (time: 13141, pfull: 6, lat: 96)> Size: 61MB
array([[[ -0.94061761,  -1.17039543,  -1.06328067, ...,   5.06540163,
           4.11665989,   1.73783432],
        [ -0.94064331,  -1.17038761,  -1.27502992, ...,   5.33837531,
           4.7691719 ,   2.26024181],
        [ -0.85740551,  -1.67704429,  -2.46299881, ...,   6.04645889,
           4.68753587,   1.91613756],
        [ -3.90448176,  -4.72385674,  -4.84902252, ...,   6.35283131,
           5.27680956,   2.537938  ],
        [ -3.70776684,  -7.00717367,  -8.50431661, ...,   6.61854361,
           6.47989733,   3.88874817],
        [ -3.3852789 ,  -7.42113157, -10.35890047, ...,   3.67831065,
           3.89067339,   2.47093434]],

       [[ -0.80040589,  -0.71147852,  -0.03758618, ...,   3.46462783,
           5.11653328,   3.29183745],
        [ -0.80041172,  -0.7114788 ,  -0.12443701, ...,   2.98776736,
           5.10214428,   3.59322859],
        [ -0.70964612,  -0.81582052,  -1.10611223, ...,   2.8921839 ,
           4.4310219 ,   3.00739832],
        [ -2.06413008,  -3.13564548,  -3.36630461, ...,   2.71913168,
...
          -5.62036125,  -1.97765324],
        [ -0.99082611,  -2.53904406,  -3.16740739, ...,  -9.38255681,
          -6.72819258,  -2.66703288],
        [ -2.60144739,  -4.39891946,  -4.28408979, ..., -11.06781885,
          -8.03384686,  -3.20223416],
        [ -2.68365436,  -5.34463268,  -6.24665776, ...,  -7.60438686,
          -7.73007421,  -4.03362861]],

       [[ -0.72744384,  -1.10809551,  -1.02496077, ...,  -0.1224137 ,
           0.56069914,   0.5443902 ],
        [ -0.72744412,  -1.10810463,  -1.01523207, ...,  -2.45476296,
          -0.2919175 ,   0.75655257],
        [ -0.82546378,  -1.54609506,  -1.87005753, ...,  -5.46711635,
          -1.12630689,   1.08819566],
        [ -1.70869964,  -2.96685635,  -3.11539521, ...,  -7.26472409,
          -1.31001465,   1.61611558],
        [ -2.93176925,  -4.79457129,  -4.25633753, ...,  -8.19860863,
          -1.38958241,   2.08674282],
        [ -3.31780965,  -5.92012959,  -6.50491182, ...,  -3.77456514,
           1.01656009,   2.40685252]]])
Coordinates:
  * pfull    (pfull) float64 48B 925.0 850.0 700.0 600.0 500.0 250.0
  * lat      (lat) float64 768B -88.57 -86.72 -84.86 -83.0 ... 84.86 86.72 88.57
  * time     (time) datetime64[ns] 105kB 1979-01-01 1979-01-02 ... 2015-01-01